# Clinical Documentation Assistant


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob

# Assuming drive is already mounted from a previous step or manually by the user

# Reset về thư mục root của Colab
%cd /content/
if not os.path.exists('Clinical-Ambient-Documentation-Assistant'):
    !git clone https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant.git

%cd /content/Clinical-Ambient-Documentation-Assistant

# Tìm file zip
possible_path = "/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/data_lake.zip"
if not os.path.exists(possible_path):
    # Fallback if the primary path is not found, search for audio_data.zip
    search = glob.glob("/content/drive/MyDrive/**/audio_data.zip", recursive=True)
    if search:
        possible_path = search[0]

if os.path.exists(possible_path):
    print(f"Đang giải nén dữ liệu từ: {possible_path}...")
    # Sử dụng tùy chọn -o để ghi đè và đảm bảo giải nén toàn bộ cấu trúc thư mục
    !unzip -o -q "{possible_path}" -d .
    print("Giải nén dữ liệu thành công!")
    # Kiểm tra lại xem thư mục audio đã xuất hiện chưa
    if os.path.exists('data/data_lake/silver/audio_clean'):
        print("Đã tìm thấy thư mục audio_clean.")
    else:
        print("CẢNH BÁO: Vẫn không thấy thư mục audio_clean sau khi giải nén!")
else:
    print("LỖI: Không tìm thấy file data_lake.zip!")

!pip install -r requirements.txt

/content
/content/Clinical-Ambient-Documentation-Assistant
Đang giải nén dữ liệu từ: /content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/data_lake.zip...
Giải nén dữ liệu thành công!
Đã tìm thấy thư mục audio_clean.


## Thiết lập môi trường


In [ ]:
import json
import os

# Đảm bảo đang ở đúng thư mục dự án
%cd /content/Clinical-Ambient-Documentation-Assistant

original_manifest = 'data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl'
fixed_manifest = 'data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl'

# Định nghĩa các prefix cũ và mới
old_prefix = '/home/duykhongngu28/massive/Clinical Ambient Documentation Assistant/'
new_prefix = '/content/Clinical-Ambient-Documentation-Assistant/'

if os.path.exists(original_manifest):
    os.makedirs(os.path.dirname(fixed_manifest), exist_ok=True)
    with open(original_manifest, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    with open(fixed_manifest, 'w', encoding='utf-8') as f:
        for line in lines:
            data = json.loads(line)
            # Cập nhật tất cả các trường có thể chứa đường dẫn
            for key in ['audio_filepath', 'raw_audio_path', 'raw_transcript_path']:
                if key in data and isinstance(data[key], str):
                    data[key] = data[key].replace(old_prefix, new_prefix)
            f.write(json.dumps(data, ensure_ascii=False) + '\n')
    print(f'Đã cập nhật manifest tại: {os.path.abspath(fixed_manifest)}')
else:
    print(f'LỖI: Không tìm thấy manifest gốc tại {original_manifest}.')

/content/Clinical-Ambient-Documentation-Assistant
Đã cập nhật manifest tại: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl


## Chuẩn bị dữ liệu


In [ ]:
!pip install --upgrade transformers accelerate librosa datasets soundfile

## Bắt đầu tiến hành chuẩn bị training model

In [ ]:
import os

# Define paths
PROJECT_ROOT = '/content/Clinical-Ambient-Documentation-Assistant'
BUNDLE_PATH = '/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/week5_safe_train_dev_audio_bundle.tar.gz'

if os.path.exists(BUNDLE_PATH):
    print(f"Extracting {BUNDLE_PATH} to {PROJECT_ROOT}...")
    # -x: extract, -z: gunzip, -f: file
    !tar -xzf "{BUNDLE_PATH}" -C "{PROJECT_ROOT}"
    print("Extraction complete!")

    # Verification
    check_dir = os.path.join(PROJECT_ROOT, 'data/data_lake/silver/audio_preprocessed/p03_vad_pad_200ms/')
    if os.path.exists(check_dir):
        print(f"Confirmed: Preprocessed audio directory found at {check_dir}")
    else:
        print("Warning: Target directory still not found. Please check the structure inside the tar.gz file.")
else:
    print(f"Error: Bundle not found at {BUNDLE_PATH}. Please ensure the file is uploaded and Drive is mounted.")

Extracting /content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/week5_safe_train_dev_audio_bundle.tar.gz to /content/Clinical-Ambient-Documentation-Assistant...
Extraction complete!
Confirmed: Preprocessed audio directory found at /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/audio_preprocessed/p03_vad_pad_200ms/


In [ ]:
import os

# Define the directory to check
data_dir = '/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/'

print(f"Listing contents of: {data_dir}")

if os.path.exists(data_dir):
    # List all files and directories recursively
    for root, dirs, files in os.walk(data_dir):
        for name in files:
            print(os.path.join(root, name))
        for name in dirs:
            print(os.path.join(root, name) + '/')
else:
    print(f"The directory '{data_dir}' does not exist. Please ensure your Google Drive is correctly mounted and the path is accurate.")

Listing contents of: /content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/
/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/data_lake.zip
/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/audio_data.zip
/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/week5_safe_train_dev_audio_bundle.tar.gz
/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/clinical_asr_transfer/
/content/drive/MyDrive/Clinical Ambient Docs Assistant Dataset/clinical_asr_transfer/week5_safe_train_dev_audio_bundle.tar.gz


In [ ]:
!python scripts/asr_eval/run_whisper_transformers.py \
  --manifest data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl \
  --model vinai/PhoWhisper-medium \
  --output experiments/asr/finetune/week5/baseline_safe_dev/phowhisper_medium_safe_dev_predictions.jsonl

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2254, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2488, in _get_module
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2486, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importl

### Kiểm tra file prediction hiện tại

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter

# Chuyển đổi sang thư mục dự án
os.chdir('/content/Clinical-Ambient-Documentation-Assistant')

pred_path = Path("experiments/asr/finetune/week5/baseline_safe_dev/phowhisper_medium_safe_dev_predictions.jsonl")
manifest_path = Path("data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl")

print("prediction_exists:", pred_path.exists())

if not pred_path.exists():
    print("[ERROR] prediction file does not exist")
else:
    pred_rows = []
    bad_lines = []

    for i, line in enumerate(pred_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            pred_rows.append(json.loads(line))
        except Exception as e:
            bad_lines.append((i, str(e), line[:200]))

    manifest_rows = [
        json.loads(x)
        for x in manifest_path.read_text(encoding="utf-8").splitlines()
        if x.strip()
    ]

    pred_ids = [r.get("sample_id") for r in pred_rows]
    manifest_ids = [r.get("sample_id") for r in manifest_rows]

    pred_counter = Counter(pred_ids)
    duplicates = [sid for sid, c in pred_counter.items() if c > 1]
    missing_from_pred = sorted(set(manifest_ids) - set(pred_ids))
    extra_in_pred = sorted(set(pred_ids) - set(manifest_ids))

    print("manifest_rows:", len(manifest_rows))
    print("prediction_rows:", len(pred_rows))
    print("unique_prediction_sample_ids:", len(set(pred_ids)))
    print("bad_json_lines:", len(bad_lines))
    print("duplicates:", len(duplicates))
    print("missing_from_prediction:", len(missing_from_pred))
    print("extra_in_prediction:", len(extra_in_pred))

    if duplicates[:10]:
        print("duplicate sample_ids:", duplicates[:10])
    if missing_from_pred[:10]:
        print("missing sample_ids:", missing_from_pred[:10])
    if extra_in_pred[:10]:
        print("extra sample_ids:", extra_in_pred[:10])
    if bad_lines[:3]:
        print("bad lines:", bad_lines[:3])

    ok = (
        len(pred_rows) == 200
        and len(set(pred_ids)) == 200
        and len(bad_lines) == 0
        and len(missing_from_pred) == 0
        and len(extra_in_pred) == 0
    )

    print("STATUS:", "PASS" if ok else "FAIL")

prediction_exists: True
manifest_rows: 200
prediction_rows: 200
unique_prediction_sample_ids: 200
bad_json_lines: 0
duplicates: 0
missing_from_prediction: 0
extra_in_prediction: 0
STATUS: PASS


In [ ]:
import os
from pathlib import Path

# Kiểm tra sự tồn tại của các file cần thiết để đóng gói
manifests = [
    "data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl",
    "data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl",
]

print("--- Kiểm tra file trước khi tải về local ---")
for m in manifests:
    path = Path('/content/Clinical-Ambient-Documentation-Assistant') / m
    print(f"Manifest: {m} -> {'✅ OK' if path.exists() else '❌ MISSING'}")

audio_dir = Path('/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/audio_preprocessed/p03_vad_pad_200ms/')
if audio_dir.exists():
    count = len(list(audio_dir.glob('*.wav')))
    print(f"Audio Dir: {audio_dir} -> ✅ Tìm thấy {count} file .wav")
else:
    print(f"Audio Dir: {audio_dir} -> ❌ KHÔNG TÌM THẤY")

--- Kiểm tra file trước khi tải về local ---
Manifest: data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl -> ✅ OK
Manifest: data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl -> ✅ OK
Audio Dir: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/audio_preprocessed/p03_vad_pad_200ms -> ✅ Tìm thấy 796 file .wav


In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path("/content/Clinical-Ambient-Documentation-Assistant")

# List of manifests to verify
manifest_files = [
    "data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl",
    "data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl",
]

for m_rel_path in manifest_files:
    manifest_path = PROJECT_ROOT / m_rel_path

    if not manifest_path.exists():
        print(f"❌ Manifest not found: {m_rel_path}")
        continue

    rows = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]

    missing = []
    found_count = 0

    for r in rows:
        # Check possible path keys in manifest
        audio_rel = r.get("preprocessed_audio_path") or r.get("clean_audio_path") or r.get("audio_filepath")

        if not audio_rel:
            missing.append((r.get("sample_id"), "No path key found in row"))
            continue

        # Build absolute path
        p = Path(audio_rel)
        if not p.is_absolute():
            p = PROJECT_ROOT / p

        if p.exists():
            found_count += 1
        else:
            missing.append((r.get("sample_id"), str(audio_rel)))

    print(f"\n--- Verification for: {m_rel_path} ---")
    print(f"Total rows: {len(rows)}")
    print(f"Files found: {found_count}")
    print(f"Files missing: {len(missing)}")

    if missing:
        print("First 5 missing samples:")
        for mid, mpath in missing[:5]:
            print(f"  - ID: {mid} | Path: {mpath}")


--- Verification for: data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl ---
Total rows: 600
Files found: 600
Files missing: 0

--- Verification for: data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl ---
Total rows: 200
Files found: 200
Files missing: 0


In [ ]:
import os

# Kiểm tra danh sách file trong scripts/asr_eval để tìm script chuẩn bị manifest
print("--- Danh sách file trong scripts/asr_eval: ---")
!ls /content/Clinical-Ambient-Documentation-Assistant/scripts/asr_eval/

# Thiết lập PROJECT_ROOT đúng
os.environ['PROJECT_ROOT'] = '/content/Clinical-Ambient-Documentation-Assistant'
%cd $PROJECT_ROOT

# Thử tìm file script có tên tương tự
import glob
scripts = glob.glob("scripts/asr_eval/prepare*manifest*.py")
if scripts:
    target_script = scripts[0]
    print(f"\n--- Đã tìm thấy script: {target_script} ---")

    # Tạo thư mục output
    !mkdir -p experiments/asr/finetune/week5/data/

    # Thực thi với script vừa tìm được
    !python3 {target_script} \
      --input_manifest data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl \
      --output_manifest experiments/asr/finetune/week5/data/train_manifest_for_phowhisper.jsonl \
      --split train \
      --project_root "$PROJECT_ROOT" \
      --expected_preprocessing p03_safe_hybrid_v0_1 \
      --require_audio_readable \
      --fail_on_skipped
else:
    print("\n❌ Không tìm thấy script chuẩn bị manifest. Vui lòng kiểm tra lại cấu trúc repo.")

--- Danh sách file trong scripts/asr_eval: ---
asr_medical_error_analysis.py
build_hf_audio_dataset_from_manifest.py
check_asr_manifest.py
check_split_leakage.py
clinical_error_analysis.py
compute_wer.py
create_asr_splits.py
filter_predictions_by_edge_risk.py
finetune_phowhisper_seq2seq.py
model_adapters
prepare_phowhisper_finetune_manifest.py
run_chunkformer_ctc.py
run_hf_ctc_asr.py
run_whisper_transformers.py
text_normalization_vi.py
/content/Clinical-Ambient-Documentation-Assistant

--- Đã tìm thấy script: scripts/asr_eval/prepare_phowhisper_finetune_manifest.py ---
{
  "input_manifest": "/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl",
  "output_manifest": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/data/train_manifest_for_phowhisper.jsonl",
  "split": "train",
  "expected_preprocessing": "p03_safe_hybrid_v0_1",
  "n_output_rows": 600,
  "n_skipped_rows":

In [ ]:
# Chuẩn bị manifest cho tập Validation (dev)
!python3 scripts/asr_eval/prepare_phowhisper_finetune_manifest.py \
  --input_manifest data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl \
  --output_manifest experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl \
  --split dev \
  --project_root "$PROJECT_ROOT" \
  --expected_preprocessing p03_safe_hybrid_v0_1 \
  --require_audio_readable \
  --fail_on_skipped

{
  "input_manifest": "/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl",
  "output_manifest": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl",
  "split": "dev",
  "expected_preprocessing": "p03_safe_hybrid_v0_1",
  "n_output_rows": 200,
  "n_skipped_rows": 0,
  "preprocessing_versions": {
    "p03_safe_hybrid_v0_1": 200
  },
  "manual_fallback_applied_count": 0,
  "duration_seconds": {
    "min": 2.0,
    "p10": 5.0,
    "median": 6.0,
    "mean": 5.9826,
    "p90": 7.0,
    "max": 8.0
  },
  "transcript_words": {
    "min": 9,
    "median": 27.0,
    "mean": 25.735,
    "max": 39
  },
  "skipped_reasons": {},
  "created_at": "2026-06-10T11:56:33.347245+00:00"
}
[DONE] output_manifest: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl
[DONE] skipped_rows:   

### Fix Audio File Paths in Manifests

The previous error indicates that the audio file paths within the manifest files (`vietmed_train_preprocessed_selected_safe_v0_1.jsonl` and `vietmed_dev_preprocessed_selected_safe_v0_1.jsonl`) are pointing to an incorrect absolute location (`/home/duykhongngu28/massive/...`). We need to update these paths to reflect the correct root directory in the Colab environment (`/content/Clinical-Ambient-Documentation-Assistant/`).

In [ ]:
import json
import os

PROJECT_ROOT = '/content/Clinical-Ambient-Documentation-Assistant'

# Define old and new prefixes for path correction
old_prefix = '/home/duykhongngu28/massive/Clinical Ambient Documentation Assistant/'
new_prefix = PROJECT_ROOT + '/'

manifests_to_fix = [
    'data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl',
    'data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl'
]

for relative_manifest_path in manifests_to_fix:
    full_manifest_path = os.path.join(PROJECT_ROOT, relative_manifest_path)

    if not os.path.exists(full_manifest_path):
        print(f"Skipping: Manifest not found at {full_manifest_path}")
        continue

    updated_lines = []
    with open(full_manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # Update all keys that might contain file paths
            for key in ['audio_filepath', 'raw_audio_path', 'preprocessed_audio_path', 'clean_audio_path']:
                if key in data and isinstance(data[key], str):
                    # Replace the old absolute path with the new one
                    if data[key].startswith(old_prefix):
                        data[key] = data[key].replace(old_prefix, new_prefix)
                    # Ensure paths are absolute if they are expected to be
                    elif not os.path.isabs(data[key]):
                        data[key] = os.path.join(new_prefix, data[key])
            updated_lines.append(json.dumps(data, ensure_ascii=False))

    with open(full_manifest_path, 'w', encoding='utf-8') as f:
        for line in updated_lines:
            f.write(line + '\n')
    print(f'Successfully updated paths in: {full_manifest_path}')

print("Path correction complete for all specified manifests.")

Successfully updated paths in: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl
Successfully updated paths in: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl
Path correction complete for all specified manifests.


### Tính WER/CER

In [ ]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/finetune/week5/baseline_safe_dev/phowhisper_medium_safe_dev_predictions.jsonl \
  --output experiments/asr/finetune/week5/baseline_safe_dev/phowhisper_medium_safe_dev_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.246357101224014,
  "normalized_wer": 0.21527103166893335,
  "strict_cer": 0.1980098133626817,
  "normalized_cer": 0.1951208327601229
}


In [ ]:
!python scripts/asr_eval/asr_medical_error_analysis.py \
  --predictions experiments/asr/finetune/week5/baseline_safe_dev/phowhisper_medium_safe_dev_predictions.jsonl \
  --terms experiments/asr/error_analysis/medical_terms_for_asr_check.json \
  --output experiments/asr/finetune/week5/baseline_safe_dev/phowhisper_medium_safe_dev_medical_errors.json

{
  "model_name": "vinai/PhoWhisper-medium",
  "n_samples": 200,
  "n_samples_with_critical_or_high_missing_error": 7,
  "groups": {
    "negation": {
      "severity": "critical",
      "samples_with_reference_terms": 52,
      "samples_with_missing_terms": 6,
      "samples_with_inserted_terms": 3,
      "missing_terms_total": 7,
      "inserted_terms_total": 3,
      "top_missing_terms": [
        [
          "không",
          5
        ],
        [
          "không có",
          2
        ]
      ],
      "top_inserted_terms": [
        [
          "không",
          3
        ]
      ]
    },
    "symptom": {
      "severity": "moderate",
      "samples_with_reference_terms": 61,
      "samples_with_missing_terms": 7,
      "samples_with_inserted_terms": 13,
      "missing_terms_total": 7,
      "inserted_terms_total": 13,
      "top_missing_terms": [
        [
          "ho",
          7
        ]
      ],
      "top_inserted_terms": [
        [
          "ho",
          13
   

In [ ]:
import os
from google.colab import files

# Đường dẫn thư mục chứa kết quả
results_dir = 'experiments/asr/finetune/week5/baseline_safe_dev/'
zip_filename = 'phowhisper_medium_evaluation_results.zip'

# Gom các file vào zip (loại bỏ đường dẫn thư mục phức tạp khi nén)
!zip -j {zip_filename} {results_dir}phowhisper_medium_safe_dev_predictions.jsonl \
                       {results_dir}phowhisper_medium_safe_dev_metrics.json \
                       {results_dir}phowhisper_medium_safe_dev_medical_errors.json

# Tải file về máy
files.download(zip_filename)

  adding: phowhisper_medium_safe_dev_predictions.jsonl (deflated 88%)
  adding: phowhisper_medium_safe_dev_metrics.json (deflated 89%)
  adding: phowhisper_medium_safe_dev_medical_errors.json (deflated 96%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python scripts/asr_eval/build_hf_audio_dataset_from_manifest.py \
  --manifest experiments/asr/finetune/week5/data/train_manifest_for_phowhisper.jsonl \
  --output_dir experiments/asr/finetune/week5/data/train_dataset_hf \
  --sampling_rate 16000

Generating train split: 600 examples [00:00, 37560.93 examples/s]
Saving the dataset (1/1 shards): 100% 600/600 [00:00<00:00, 1342.53 examples/s]
Dataset({
    features: ['sample_id', 'audio', 'sentence', 'language', 'task', 'split', 'source_name', 'dataset_id', 'source_type', 'domain', 'preprocessing_version', 'manual_fallback_applied', 'source_audio_field', 'source_transcript_field', 'train_allowed', 'split_role', 'duration_seconds', 'sample_rate_hz', 'channels', 'audio_format', 'audio_subtype', 'transcript_chars', 'transcript_words', 'created_at', 'created_by', 'manifest_version', 'original_duration_seconds', 'processed_duration_seconds', 'duration_delta_seconds', 'preprocessed_checksum_sha256', 'source_clean_audio_path', 'manual_fallback_reason', 'fallback_from_preprocessing'],
    num_rows: 600
})
[DONE] saved to: experiments/asr/finetune/week5/data/train_dataset_hf


In [ ]:
!python scripts/asr_eval/build_hf_audio_dataset_from_manifest.py \
  --manifest experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl \
  --output_dir experiments/asr/finetune/week5/data/dev_dataset_hf \
  --sampling_rate 16000

Generating train split: 200 examples [00:00, 27937.81 examples/s]
Saving the dataset (1/1 shards): 100% 200/200 [00:00<00:00, 1425.28 examples/s]
Dataset({
    features: ['sample_id', 'audio', 'sentence', 'language', 'task', 'split', 'source_name', 'dataset_id', 'source_type', 'domain', 'preprocessing_version', 'manual_fallback_applied', 'source_audio_field', 'source_transcript_field', 'train_allowed', 'split_role', 'duration_seconds', 'sample_rate_hz', 'channels', 'audio_format', 'audio_subtype', 'transcript_chars', 'transcript_words', 'created_at', 'created_by', 'manifest_version', 'original_duration_seconds', 'processed_duration_seconds', 'duration_delta_seconds', 'preprocessed_checksum_sha256', 'source_clean_audio_path'],
    num_rows: 200
})
[DONE] saved to: experiments/asr/finetune/week5/data/dev_dataset_hf


### Re-applying path corrections and regenerating dataset

The `FileNotFoundError` indicates that the path corrections applied previously were not effective for the `dev_manifest_for_phowhisper.jsonl` used to build the Hugging Face dataset. This is likely due to the order of operations: the `dev_manifest_for_phowhisper.jsonl` was generated *before* the final path correction on its source manifest.

I will perform the following steps:
1.  **Re-run path correction**: Ensure that `vietmed_train_preprocessed_selected_safe_v0_1.jsonl` and `vietmed_dev_preprocessed_selected_safe_v0_1.jsonl` have their paths correctly updated.
2.  **Regenerate `dev_manifest_for_phowhisper.jsonl`**: Create this manifest again, ensuring it uses the now-corrected paths from `vietmed_dev_preprocessed_selected_safe_v0_1.jsonl`.
3.  **Rebuild Hugging Face dataset**: Attempt to build the dataset with the corrected manifest.

In [ ]:
import json
import os

PROJECT_ROOT = '/content/Clinical-Ambient-Documentation-Assistant'

# Define old and new prefixes for path correction
old_prefix = '/home/duykhongngu28/massive/Clinical Ambient Documentation Assistant/'
new_prefix = PROJECT_ROOT + '/'

manifests_to_fix = [
    'data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl',
    'data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl'
]

for relative_manifest_path in manifests_to_fix:
    full_manifest_path = os.path.join(PROJECT_ROOT, relative_manifest_path)

    if not os.path.exists(full_manifest_path):
        print(f"Skipping: Manifest not found at {full_manifest_path}")
        continue

    updated_lines = []
    with open(full_manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # Update all keys that might contain file paths
            for key in ['audio_filepath', 'raw_audio_path', 'preprocessed_audio_path', 'clean_audio_path']:
                if key in data and isinstance(data[key], str):
                    # Replace the old absolute path with the new one
                    if data[key].startswith(old_prefix):
                        data[key] = data[key].replace(old_prefix, new_prefix)
                    # Ensure paths are absolute if they are expected to be
                    elif not os.path.isabs(data[key]):
                        data[key] = os.path.join(new_prefix, data[key])
            updated_lines.append(json.dumps(data, ensure_ascii=False))

    with open(full_manifest_path, 'w', encoding='utf-8') as f:
        for line in updated_lines:
            f.write(line + '\n')
    print(f'Successfully re-updated paths in: {full_manifest_path}')

print("Path correction complete for all specified manifests.")

Successfully re-updated paths in: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/vietmed_train_preprocessed_selected_safe_v0_1.jsonl
Successfully re-updated paths in: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl
Path correction complete for all specified manifests.


### Regenerating `dev_manifest_for_phowhisper.jsonl`

Now that the source manifest (`vietmed_dev_preprocessed_selected_safe_v0_1.jsonl`) has been thoroughly corrected, we need to regenerate the specific manifest for fine-tuning (`dev_manifest_for_phowhisper.jsonl`) to ensure it contains the correct paths.

In [ ]:
# Ensure PROJECT_ROOT is set if not already
# os.environ['PROJECT_ROOT'] = '/content/Clinical-Ambient-Documentation-Assistant'
# %cd $PROJECT_ROOT # This is already handled at the beginning of the notebook

# Chuẩn bị manifest cho tập Validation (dev)
!python3 scripts/asr_eval/prepare_phowhisper_finetune_manifest.py \
  --input_manifest data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl \
  --output_manifest experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl \
  --split dev \
  --project_root "$PROJECT_ROOT" \
  --expected_preprocessing p03_safe_hybrid_v0_1 \
  --require_audio_readable \
  --fail_on_skipped

{
  "input_manifest": "/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/vietmed_dev_preprocessed_selected_safe_v0_1.jsonl",
  "output_manifest": "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl",
  "split": "dev",
  "expected_preprocessing": "p03_safe_hybrid_v0_1",
  "n_output_rows": 200,
  "n_skipped_rows": 0,
  "preprocessing_versions": {
    "p03_safe_hybrid_v0_1": 200
  },
  "manual_fallback_applied_count": 0,
  "duration_seconds": {
    "min": 2.0,
    "p10": 5.0,
    "median": 6.0,
    "mean": 5.9826,
    "p90": 7.0,
    "max": 8.0
  },
  "transcript_words": {
    "min": 9,
    "median": 27.0,
    "mean": 25.735,
    "max": 39
  },
  "skipped_reasons": {},
  "created_at": "2026-06-10T11:56:40.049925+00:00"
}
[DONE] output_manifest: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl
[DONE] skipped_rows:   

### Rebuilding Hugging Face Dataset

1.   Mục danh sách
2.   Mục danh sách



With the manifests now correctly updated, we can proceed to build the Hugging Face audio dataset.

In [ ]:
# !python scripts/asr_eval/build_hf_audio_dataset_from_manifest.py \
#   --manifest experiments/asr/finetune/week5/data/dev_manifest_for_phowhisper.jsonl \
#   --output_dir experiments/asr/finetune/week5/data/dev_dataset_hf \
#   --sampling_rate 16000

Generating train split: 200 examples [00:00, 17315.74 examples/s]
Saving the dataset (1/1 shards): 100% 200/200 [00:00<00:00, 959.96 examples/s] 
Dataset({
    features: ['sample_id', 'audio', 'sentence', 'language', 'task', 'split', 'source_name', 'dataset_id', 'source_type', 'domain', 'preprocessing_version', 'manual_fallback_applied', 'source_audio_field', 'source_transcript_field', 'train_allowed', 'split_role', 'duration_seconds', 'sample_rate_hz', 'channels', 'audio_format', 'audio_subtype', 'transcript_chars', 'transcript_words', 'created_at', 'created_by', 'manifest_version', 'original_duration_seconds', 'processed_duration_seconds', 'duration_delta_seconds', 'preprocessed_checksum_sha256', 'source_clean_audio_path'],
    num_rows: 200
})
[DONE] saved to: experiments/asr/finetune/week5/data/dev_dataset_hf


## Chạy debug trước khi chạy fullrun 01

### Update Code from Git

To ensure you have the latest code without re-running the model predictions, execute the cell below. It will navigate to your project directory and pull any new changes from the `main` branch.

In [ ]:
import os

repo_path = '/content/Clinical-Ambient-Documentation-Assistant'

if not os.path.exists(repo_path):
    print(f"Repository not found. Cloning...")
    %cd /content/
    !git clone https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant.git
    %cd {repo_path}
else:
    %cd {repo_path}
    print("Cập nhật bản mới nhất từ Git (Ghi đè bản cục bộ)... ")
    !git fetch origin main
    !git reset --hard origin/main

print("Đã đồng bộ thành công với Git.")

/content/Clinical-Ambient-Documentation-Assistant
Cập nhật bản mới nhất từ Git (Ghi đè bản cục bộ)... 
From https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant
 * branch            main       -> FETCH_HEAD
HEAD is now at 1a73f30 Fix AudioDecoder JSON serialization error
Đã đồng bộ thành công với Git.


In [ ]:
!pip uninstall -y torchvision
!pip install torchvision


Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 60.6 MB/s eta 0:00:00


In [ ]:
# !python scripts/asr_eval/finetune_phowhisper_seq2seq.py \
# --config experiments/asr/finetune/week5/configs/run_01_phowhisper_medium_conservative.json \
# --max_train_samples 20 \
# --max_eval_samples 20 \
# --debug_max_steps 5

[INFO] Loading processor/model: vinai/PhoWhisper-medium
preprocessor_config.json: 100% 339/339 [00:00<00:00, 1.54MB/s]
config.json: 100% 1.33k/1.33k [00:00<00:00, 2.24MB/s]
tokenizer_config.json: 100% 805/805 [00:00<00:00, 5.12MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 18.7MB/s]
tokenizer.json: 100% 2.20M/2.20M [00:00<00:00, 20.6MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 17.1MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 20.7MB/s]
added_tokens.json: 100% 2.08k/2.08k [00:00<00:00, 8.23MB/s]
special_tokens_map.json: 100% 2.08k/2.08k [00:00<00:00, 7.74MB/s]
pytorch_model.bin: 100% 3.06G/3.06G [00:26<00:00, 117MB/s]
Loading weights: 100% 948/948 [00:00<00:00, 32377.90it/s]
generation_config.json: 100% 3.72k/3.72k [00:00<00:00, 9.89MB/s]
Generating train split: 600 examples [00:00, 53262.13 examples/s]
Map: 100% 600/600 [00:00<00:00, 4180.93 examples/s]
Generating train split: 200 examples [00:00, 31273.94 examples/s]
Map: 100% 200/200 [00:00<00:00, 3614.35 examples/s]
[INFO]

### Xóa thư mục kết quả của lần chạy debug

###  Full Fine-tuning Run
Sau khi debug pass, chúng ta sẽ xóa kết quả debug cũ và tiến hành huấn luyện trên toàn bộ tập dữ liệu đã chuẩn bị.

In [ ]:
!rm -rf "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative"


In [ ]:
# Xóa output của lần chạy debug để tránh xung đột hoặc ghi đè không sạch
import shutil
import os

output_dir = "experiments/asr/finetune/week5/outputs/run_01"
if os.path.exists(output_dir):
    print(f"Đang xóa thư mục debug cũ: {output_dir}")
    shutil.rmtree(output_dir)
    print("Đã dọn dẹp xong.")

In [ ]:
! HF_TOKEN="<YOUR_HF_TOKEN>" python scripts/asr_eval/finetune_phowhisper_seq2seq.py \
--config experiments/asr/finetune/week5/configs/run_01_phowhisper_medium_conservative.json

[INFO] Loading processor/model: vinai/PhoWhisper-medium
preprocessor_config.json: 100% 339/339 [00:00<00:00, 1.20MB/s]
config.json: 100% 1.33k/1.33k [00:00<00:00, 303kB/s]
tokenizer_config.json: 100% 805/805 [00:00<00:00, 2.71MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 9.10MB/s]
tokenizer.json: 100% 2.20M/2.20M [00:00<00:00, 21.8MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 12.6MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 17.3MB/s]
added_tokens.json: 100% 2.08k/2.08k [00:00<00:00, 5.81MB/s]
special_tokens_map.json: 100% 2.08k/2.08k [00:00<00:00, 4.73MB/s]
pytorch_model.bin: 100% 3.06G/3.06G [00:30<00:00, 101MB/s]
Loading weights: 100% 948/948 [00:00<00:00, 31388.74it/s]
generation_config.json: 100% 3.72k/3.72k [00:00<00:00, 7.97MB/s]
Generating train split: 600 examples [00:00, 39655.57 examples/s]
Map: 100% 600/600 [00:00<00:00, 2630.97 examples/s]
Generating train split: 200 examples [00:00, 20566.87 examples/s]
Map: 100% 200/200 [00:00<00:00, 2908.65 examples/s]
[INFO] 

### Tính lại WER/CER bằng script chuẩn của project

In [ ]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/predictions_dev.jsonl \
  --output experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/metrics_dev_project_wer.json

{
  "n_samples": 200,
  "strict_wer": 0.1657276083155236,
  "normalized_wer": 0.1657276083155236,
  "strict_cer": 0.14642087403127435,
  "normalized_cer": 0.14642087403127435
}


### Chạy medical error analysis cho run 01

In [ ]:
!python scripts/asr_eval/asr_medical_error_analysis.py \
  --predictions experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/predictions_dev.jsonl \
  --terms experiments/asr/error_analysis/medical_terms_for_asr_check.json \
  --output experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/medical_errors_dev.json

{
  "model_name": "vinai/PhoWhisper-medium",
  "n_samples": 200,
  "n_samples_with_critical_or_high_missing_error": 9,
  "groups": {
    "negation": {
      "severity": "critical",
      "samples_with_reference_terms": 52,
      "samples_with_missing_terms": 7,
      "samples_with_inserted_terms": 2,
      "missing_terms_total": 8,
      "inserted_terms_total": 2,
      "top_missing_terms": [
        [
          "không",
          6
        ],
        [
          "không có",
          2
        ]
      ],
      "top_inserted_terms": [
        [
          "không",
          1
        ],
        [
          "không có",
          1
        ]
      ]
    },
    "symptom": {
      "severity": "moderate",
      "samples_with_reference_terms": 61,
      "samples_with_missing_terms": 7,
      "samples_with_inserted_terms": 2,
      "missing_terms_total": 7,
      "inserted_terms_total": 2,
      "top_missing_terms": [
        [
          "ho",
          7
        ]
      ],
      "top_inserted

In [ ]:
import os

# Check if the file exists at the expected path
drive_zip_path = '/content/drive/MyDrive/phowhisper_run01_full_results.zip'

print(f"Checking for file: {drive_zip_path}")
if os.path.exists(drive_zip_path):
    print(f"✅ File found! Size: {os.path.getsize(drive_zip_path) / (1024*1024):.2f} MB")
else:
    print("❌ File NOT found at the specific path.")

print("\n--- Listing files in MyDrive to help locate it: ---")
!ls -lh /content/drive/MyDrive/ | grep .zip

Checking for file: /content/drive/MyDrive/phowhisper_run01_full_results.zip
✅ File found! Size: 18859.72 MB

--- Listing files in MyDrive to help locate it: ---
-rw-------  1 root root  19G Jun 10 13:54 phowhisper_run01_full_results.zip


### 📦 Sao lưu kết quả huấn luyện (Run 01) lên Google Drive
Cell này sẽ nén thư mục `run_01_phowhisper_medium_conservative` và lưu vào Drive của bạn.

In [ ]:
import os

# Đường dẫn kết quả vừa chạy xong
results_path = "experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative"
# Đường dẫn lưu trên Drive
zip_on_drive = "/content/drive/MyDrive/phowhisper_run01_full_results.zip"

if os.path.exists(results_path):
    print(f"--- Đang nén kết quả từ: {results_path} ---")
    # Nén thư mục và lưu thẳng vào Drive
    !zip -r "{zip_on_drive}" "{results_path}"

    if os.path.exists(zip_on_drive):
        print(f"\n✅ Thành công! File đã được lưu tại: {zip_on_drive}")
        print(f"Kích thước: {os.path.getsize(zip_on_drive) / (1024*1024):.2f} MB")
    else:
        print("\n❌ Có lỗi xảy ra, không tìm thấy file zip trên Drive.")
else:
    print(f"❌ LỖI: Không tìm thấy thư mục {results_path}. Hãy kiểm tra lại tên thư mục.")

--- Đang nén kết quả từ: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative ---
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/ (stored 0%)
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/medical_errors_dev.json (deflated 96%)
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/predictions_dev.jsonl (deflated 88%)
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/RUN_01_REPORT.md (deflated 35%)
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/metrics_dev_project_wer.json (deflated 89%)
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model/ (stored 0%)
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model/model.safetensors (deflated 7%)
  adding: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model/tokenizer.json (deflated 81

In [ ]:
!unzip /content/drive/MyDrive/phowhisper_run01_full_results.zip

Archive:  /content/drive/MyDrive/phowhisper_run01_full_results.zip
   creating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/
  inflating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/medical_errors_dev.json  
  inflating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/predictions_dev.jsonl  
  inflating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/RUN_01_REPORT.md  
  inflating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/metrics_dev_project_wer.json  
   creating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model/
  inflating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model/model.safetensors  
  inflating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model/tokenizer.json  
  inflating: experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model/generation_c

In [ ]:
token = userdata.get('HF_TOKEN')
print(token)

<YOUR_HF_TOKEN>


In [ ]:
import os
from huggingface_hub import HfApi, create_repo
from google.colab import userdata
import glob

# 1. Cấu hình
token = userdata.get('HF_TOKEN')
repo_id = "DukeShy/phowhisper-medium-medical-vi-run01"

# Tự động tìm đường dẫn đến thư mục best_model
search_pattern = "/content/**/run_01_phowhisper_medium_conservative/best_model"
found_paths = glob.glob(search_pattern, recursive=True)

if found_paths:
    model_dir = found_paths[0]
    api = HfApi(token=token)

    try:
        # 2. Tạo repository nếu chưa có
        print(f"🔨 Đang kiểm tra/tạo repository: {repo_id}")
        create_repo(repo_id=repo_id, token=token, private=True, exist_ok=True)

        # 3. Upload folder
        print(f"🚀 Đang upload model từ {model_dir}...")
        api.upload_folder(
            folder_path=model_dir,
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload fine-tuned best model (run_01)"
        )
        print(f"\n🎉 Thành công! Xem tại: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"❌ Lỗi: {e}")
        print("\n💡 Gợi ý: Hãy kiểm tra xem HF_TOKEN có quyền 'Write' không.")
else:
    print("❌ LỖI: Không tìm thấy thư mục 'best_model'.")

🔨 Đang kiểm tra/tạo repository: DukeShy/phowhisper-medium-medical-vi-run01
🚀 Đang upload model từ /content/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/best_model...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t_model/model.safetensors:   0%|          | 99.6kB / 3.06GB            

  ...t_model/training_args.bin:  10%|#         |   561B / 5.46kB            


🎉 Thành công! Xem tại: https://huggingface.co/DukeShy/phowhisper-medium-medical-vi-run01


In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata
import os

# Cấu hình
token = userdata.get('HF_TOKEN')
repo_id = "DukeShy/phowhisper-medium-medical-vi-run01"
results_dir = "/content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative"

files_to_upload = [
    "predictions_dev.jsonl",
    "metrics_dev_project_wer.json",
    "medical_errors_dev.json"
]

api = HfApi(token=token)

print(f"🚀 Đang upload các file bổ sung lên {repo_id}...")

for file_name in files_to_upload:
    file_path = os.path.join(results_dir, file_name)
    if os.path.exists(file_path):
        print(f"Uploading: {file_name}")
        api.upload_file(
            path_or_fileobj=file_path,
            path_in_repo=file_name,
            repo_id=repo_id,
            repo_type="model",
            commit_message=f"Upload {file_name} evaluation results"
        )
    else:
        print(f"❌ Không tìm thấy file: {file_path}")

print("\n🎉 Đã hoàn tất upload các file bổ sung!")

🚀 Đang upload các file bổ sung lên DukeShy/phowhisper-medium-medical-vi-run01...
❌ Không tìm thấy file: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/predictions_dev.jsonl
❌ Không tìm thấy file: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/metrics_dev_project_wer.json
❌ Không tìm thấy file: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/medical_errors_dev.json

🎉 Đã hoàn tất upload các file bổ sung!


In [ ]:
import os
import glob
from huggingface_hub import HfApi
from google.colab import userdata

# 1. Cấu hình
token = userdata.get('HF_TOKEN')
repo_id = "DukeShy/phowhisper-medium-medical-vi-run01"
api = HfApi(token=token)

# Danh sách file mục tiêu
target_files = [
    "predictions_dev.jsonl",
    "metrics_dev_project_wer.json",
    "medical_errors_dev.json"
]

print(f"🔍 Đang tìm kiếm các file đánh giá trong /content/...")

found_files = {}
for file_name in target_files:
    # Tìm kiếm file trong toàn bộ thư mục content
    search_pattern = f"/content/**/{file_name}"
    matches = glob.glob(search_pattern, recursive=True)
    if matches:
        # Ưu tiên path nằm trong folder run_01
        run_01_matches = [m for m in matches if "run_01" in m]
        found_files[file_name] = run_01_matches[0] if run_01_matches else matches[0]
        print(f"✅ Tìm thấy {file_name} tại: {found_files[file_name]}")
    else:
        print(f"❌ Không tìm thấy file: {file_name}")

if not found_files:
    print("\n🛑 LỖI: Không tìm thấy bất kỳ file nào để upload. Vui lòng đảm bảo bạn đã giải nén thành công.")
else:
    print(f"\n🚀 Bắt đầu upload lên repo: {repo_id}")
    success_count = 0
    for file_name, local_path in found_files.items():
        try:
            print(f"📤 Đang upload: {file_name}...")
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=file_name,
                repo_id=repo_id,
                repo_type="model",
                commit_message=f"Add {file_name} evaluation results"
            )
            success_count += 1
            print(f"✨ Thành công: {file_name}")
        except Exception as e:
            print(f"❌ Lỗi khi upload {file_name}: {e}")

    print(f"\n🎉 Hoàn tất! Đã upload {success_count}/{len(target_files)} file.")

🔍 Đang tìm kiếm các file đánh giá trong /content/...
✅ Tìm thấy predictions_dev.jsonl tại: /content/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/predictions_dev.jsonl
✅ Tìm thấy metrics_dev_project_wer.json tại: /content/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/metrics_dev_project_wer.json
✅ Tìm thấy medical_errors_dev.json tại: /content/experiments/asr/finetune/week5/run_01_phowhisper_medium_conservative/medical_errors_dev.json

🚀 Bắt đầu upload lên repo: DukeShy/phowhisper-medium-medical-vi-run01
📤 Đang upload: predictions_dev.jsonl...
✨ Thành công: predictions_dev.jsonl
📤 Đang upload: metrics_dev_project_wer.json...
✨ Thành công: metrics_dev_project_wer.json
📤 Đang upload: medical_errors_dev.json...
✨ Thành công: medical_errors_dev.json

🎉 Hoàn tất! Đã upload 3/3 file.
